In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import re

In [2]:
# Step 1: Fetch the page content
url = "https://en.wikipedia.org/wiki/List_of_cinematic_firsts"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

output_file = 'cinema_timeline.csv'

# Focus on the main content section (the timeline events are within mw-content-text)
content_div = soup.find('div', {'id': 'mw-content-text'})

# Open the output CSV file to write the cleaned data
with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Date', 'Event Description'])

    events = []
    current_year = None
    start_processing = False

    # Begin parsing within the content div
    for section in content_div.find_all(['h2', 'h3', 'p', 'ul']):
        # Stop parsing if we reach sections like "See also" or "References"
        if 'See also' in section.text or 'Further reading' in section.text or 'References' in section.text or 'Bibliography' in section.text:
            break

        # Extract potential decade or specific year from headings (like <h2>, <h3>, etc.)
        if section.name in ['h2', 'h3']:
            heading_text = section.text.strip()

            # Skip irrelevant century and decade headings
            if 'century' in heading_text.lower() or 'decade' in heading_text.lower():
                continue

            # If it's a valid year, assign it as current_year and start processing
            match = re.search(r'(\d{4})', heading_text)
            if match:
                current_year = match.group(1)
                start_processing = True
                print(f"Processing year: {current_year}")

        # Only process sections after encountering the first valid year
        if start_processing and section.name == 'ul':  # Events are often listed in bullet points
            for li in section.find_all('li'):
                event_text = li.text.strip()
                if ": " in event_text:
                    # Split to remove month/day, keep only event description
                    #event_description = event_text.split(": ", 1)[1]
                    # Remove double quotes
                    event_descriptio = re.sub(r'"', '', event_description).strip()
                    # Remove reference notes (e.g., [1])
                    event_description = re.sub(r'\[\d+\]', '', event_description).strip()
                    events.append((current_year, event_description))

    # Write the events to the CSV file
    writer.writerows(events)

print("CSV file created with cinema events.")

CSV file created with cinema events.


In [5]:
import requests
from bs4 import BeautifulSoup
import csv
import re

# Step 1: Fetch the page content
url = "https://en.wikipedia.org/wiki/List_of_cinematic_firsts"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

output_file = 'cinema_timeline.csv'

# Focus on the main content section (the timeline events are within mw-content-text)
content_div = soup.find('div', {'id': 'mw-content-text'})

# Open the output CSV file to write the cleaned data
with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['Date', 'Event Description'])

    events = []
    current_year = None

    # Begin parsing within the content div
    for section in content_div.find_all(['h2', 'h3', 'ul']):
        # Stop parsing if we reach sections like "See also" or "References"
        if 'See also' in section.text or 'Further reading' in section.text or 'References' in section.text or 'Bibliography' in section.text:
            break

        # Extract specific year from headings (like <h2>, <h3>, etc.)
        if section.name in ['h2', 'h3']:
            heading_text = section.text.strip()

            # Skip decade headings by checking if it contains "s" (e.g., "1870s")
            if 'century' in heading_text.lower() or 'decade' in heading_text.lower() or re.search(r'\b\d{3}0s\b', heading_text) or re.search('\b\d{4}th\b', heading_text):
                current_year = None  # Reset current_year if it's a decade heading
                continue

            # If it's a valid year, set it as current_year
            match = re.search(r'\b(\d{4})\b', heading_text)
            if match:
                current_year = match.group(1)
                print(f"Processing year: {current_year}")

        # Only process <ul> sections after encountering a valid year
        if current_year and section.name == 'ul':  # Events are often listed in bullet points
            for li in section.find_all('li'):
                event_text = li.text.strip()

                # Remove reference notes (e.g., [1]) and any other square-bracketed text
                event_description = re.sub(r'\[\d+\]', '', event_text).strip()

                # Append the event with the current year
                events.append((current_year, event_description))

    # Write the events to the CSV file
    writer.writerows(events)

print("CSV file created with cinema events.")



Processing year: 1870
CSV file created with cinema events.
